# 05. 멀티모달 인덱싱 — PDF/PPTX 업로드 및 인덱서 실행

이 노트북에서는:
1. 로컬 PDF/PPTX 파일을 Blob Storage에 업로드합니다.
2. 사전 등록된 2개의 인덱서를 실행하여 각각 다른 방식으로 인덱싱합니다.

## 두 가지 멀티모달 파이프라인

| 파이프라인 | 인덱서 | 스킬 체인 | Function App |
|-----------|--------|-----------|-------------|
| **Basic** | `st-multimodal-basic-indexer` | DI Layout (Cross-Region EUS2) → Native SplitSkill (markdown mode) → Embedding → Index | ❌ 불필요 |
| **Verbalized** | `st-multimodal-verbalized-indexer` | DI Layout (EUS2) → Custom WebApiSkill (GPT-5.4 Verbalize) → Custom WebApiSkill (markdown_split) → Embedding → Index | ✅ skills-function |

## 사전 요구사항
- `01-infra-deployment.ipynb` 실행 완료 (인덱서 사전 등록 포함)
- `.env` 파일 생성 완료
- Blob Storage Private Endpoint 승인 완료

## 데이터 흐름
```
[로컬 PDF/PPTX]
    │  업로드
    ▼
[Blob Storage — raw-documents/raw/pdf/st/]
    │  AI Search Indexer 실행
    ├────────────────────────────────────────┐
    ▼ [B] Basic                              ▼ [C] Verbalized
[DI Layout → SplitSkill → Embedding]   [DI Layout → GPT-5.4 → Split → Embedding]
    │                                        │
    ▼                                        ▼
[st-multimodal-basic-index]             [st-multimodal-verbalized-index]
```

## 1. 환경 설정

In [10]:
import os
import sys
import json
import subprocess
import time
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

# 필수 환경변수 확인
required_vars = [
    'AZURE_SEARCH_SERVICE_ENDPOINT',
    'AZURE_STORAGE_ACCOUNT_NAME',
    'AZURE_STORAGE_RESOURCE_ID',
    'AZURE_RESOURCE_GROUP',
]
for var in required_vars:
    assert os.environ.get(var), f'{var} 환경변수가 설정되지 않았습니다.'

STORAGE_NAME = os.environ['AZURE_STORAGE_ACCOUNT_NAME']
SEARCH_ENDPOINT = os.environ['AZURE_SEARCH_SERVICE_ENDPOINT']
RG_NAME = os.environ['AZURE_RESOURCE_GROUP']

# 인덱서/인덱스 이름 (setup_ai_search_multimodal_pipeline.py 와 동일)
SOURCE = 'st'  # source prefix
PDF_INDEXER = f'{SOURCE}-multimodal-pdf-indexer'           # [B-PDF] basic
PPTX_INDEXER = f'{SOURCE}-multimodal-pptx-indexer'         # [B-PPTX] basic
VERBALIZED_INDEXER = f'{SOURCE}-multimodal-verbalized-indexer'  # [C] verbalized
PDF_INDEX = f'{SOURCE}-multimodal-pdf-index'
PPTX_INDEX = f'{SOURCE}-multimodal-pptx-index'
VERBALIZED_INDEX = f'{SOURCE}-multimodal-verbalized-index'

# Blob 경로 (setup script datasource prefix='raw/' → pdf/pptx 서브폴더 모두 감시)
CONTAINER_NAME = os.environ.get('AZURE_STORAGE_CONTAINER_NAME', 'raw-documents')
BLOB_PREFIX_PDF = 'raw/pdf/'
BLOB_PREFIX_PPTX = 'raw/pptx/'

print('환경 설정 완료')
print(f'  Storage   : {STORAGE_NAME}')
print(f'  AI Search : {SEARCH_ENDPOINT}')
print(f'  Container : {CONTAINER_NAME}')
print(f'    PDF     : {BLOB_PREFIX_PDF}')
print(f'    PPTX    : {BLOB_PREFIX_PPTX}')
print(f'  Indexers  : {PDF_INDEXER}, {PPTX_INDEXER}, {VERBALIZED_INDEXER}')


환경 설정 완료
  Storage   : stragi63325wdoby
  AI Search : https://search-ragi-63325wdo.search.windows.net
  Container : raw-documents
    PDF     : raw/pdf/
    PPTX    : raw/pptx/
  Indexers  : st-multimodal-pdf-indexer, st-multimodal-pptx-indexer, st-multimodal-verbalized-indexer


## 2. Blob Storage 파일 확인

seed 스크립트로 이미 업로드된 PDF/PPTX 파일 현황을 확인합니다.

In [11]:
from azure.identity import (
    AzureCliCredential,
    DefaultAzureCredential,
    ChainedTokenCredential,
)
from azure.storage.blob import BlobServiceClient
from azure.core.exceptions import ResourceExistsError

# Azure VM 환경에서는 VM Managed Identity 의 테넌트가 스토리지 계정 테넌트와
# 다를 수 있어 'Issuer validation failed' 가 발생합니다.
# → az CLI 로 로그인한 (올바른 테넌트의) 자격을 우선 사용하도록 ChainedTokenCredential 구성.
TENANT_ID = os.environ.get('AZURE_TENANT_ID')
cli_kwargs = {'tenant_id': TENANT_ID} if TENANT_ID else {}
credential = ChainedTokenCredential(
    AzureCliCredential(**cli_kwargs),
    DefaultAzureCredential(
        exclude_managed_identity_credential=True,
        **({'interactive_browser_tenant_id': TENANT_ID,
            'shared_cache_tenant_id': TENANT_ID,
            'visual_studio_code_tenant_id': TENANT_ID,
            'workload_identity_tenant_id': TENANT_ID} if TENANT_ID else {}),
    ),
)
print(f'  Tenant   : {TENANT_ID or "(default)"}')

blob_service = BlobServiceClient(
    account_url=f'https://{STORAGE_NAME}.blob.core.windows.net',
    credential=credential,
)

container_client = blob_service.get_container_client(CONTAINER_NAME)

# 컨테이너 존재 확인 / 없으면 생성
try:
    container_client.get_container_properties()
    print(f'✅ 컨테이너 확인: {CONTAINER_NAME}')
except Exception:
    try:
        container_client.create_container()
        print(f'컨테이너 생성: {CONTAINER_NAME}')
    except ResourceExistsError:
        print(f'✅ 컨테이너 이미 존재: {CONTAINER_NAME}')


  Tenant   : (default)


✅ 컨테이너 확인: raw-documents


In [12]:
# ── Blob Storage에 업로드된 파일 현황 확인 (PDF + PPTX) ────────────────────
total_files = 0
total_bytes = 0
for prefix, label in [(BLOB_PREFIX_PDF, 'PDF'), (BLOB_PREFIX_PPTX, 'PPTX')]:
    blobs = list(container_client.list_blobs(name_starts_with=prefix))
    prefix_bytes = sum(b.size for b in blobs)
    total_files += len(blobs)
    total_bytes += prefix_bytes
    print(f'[{label}] {CONTAINER_NAME}/{prefix} — {len(blobs)}개 파일 ({prefix_bytes / (1024*1024):.1f} MiB)')
    for b in blobs:
        print(f'  {b.name}  ({b.size:,} bytes)')

print(f'\n총 {total_files}개 파일, {total_bytes / (1024*1024):.1f} MiB')
if total_files == 0:
    print('⚠️ 파일이 없습니다. seed 스크립트를 먼저 실행하세요.')


[PDF] raw-documents/raw/pdf/ — 15개 파일 (19.8 MiB)
  raw/pdf/HA/HA_0032_0013106.pdf  (706,212 bytes)
  raw/pdf/HA/HA_0051_0014672.pdf  (7,980,110 bytes)
  raw/pdf/HA/HA_0078_0044181.pdf  (1,041,389 bytes)
  raw/pdf/HA/HA_0114_0043314.pdf  (963,958 bytes)
  raw/pdf/HA/HA_0132_0067633.pdf  (735,073 bytes)
  raw/pdf/SS/SS_0017_0082677.pdf  (805,490 bytes)
  raw/pdf/SS/SS_0025_0027983.pdf  (705,201 bytes)
  raw/pdf/SS/SS_0050_0016707.pdf  (506,171 bytes)
  raw/pdf/SS/SS_0132_0068276.pdf  (615,241 bytes)
  raw/pdf/SS/SS_0144_0061959.pdf  (860,977 bytes)
  raw/pdf/ST/ST_0028_0008931.pdf  (308,419 bytes)
  raw/pdf/ST/ST_0028_0028442.pdf  (470,856 bytes)
  raw/pdf/ST/ST_0119_0006320.pdf  (3,072,631 bytes)
  raw/pdf/ST/ST_0145_0074863.pdf  (1,011,062 bytes)
  raw/pdf/ST/ST_0145_0075608.pdf  (1,018,619 bytes)
[PPTX] raw-documents/raw/pptx/ — 15개 파일 (7.4 MiB)
  raw/pptx/HA/HA_0032_0014125.pptx  (26,751 bytes)
  raw/pptx/HA/HA_0047_0038756.pptx  (27,654 bytes)
  raw/pptx/HA/HA_0077_0020961.pptx  (24

## 3. 사전 등록된 인덱서 확인

노트북 01에서 등록한 2개의 멀티모달 인덱서가 존재하는지 확인합니다.

In [13]:
import requests

API_VERSION = '2024-11-01-preview'

def get_search_headers():
    token = credential.get_token('https://search.azure.com/.default').token
    return {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

def check_indexer(indexer_name: str) -> dict | None:
    """인덱서 존재 여부 확인"""
    url = f'{SEARCH_ENDPOINT}/indexers/{indexer_name}?api-version={API_VERSION}'
    resp = requests.get(url, headers=get_search_headers())
    if resp.status_code == 200:
        return resp.json()
    return None

# 인덱서 확인
INDEXERS = [PDF_INDEXER, PPTX_INDEXER, VERBALIZED_INDEXER]
for name in INDEXERS:
    indexer = check_indexer(name)
    if indexer:
        print(f'✅ {name} — 등록됨 (skillset: {indexer.get("skillsetName", "N/A")})')
    else:
        print(f'❌ {name} — 미등록! scripts/setup_ai_search_multimodal_pipeline.py 실행 필요')


✅ st-multimodal-pdf-indexer — 등록됨 (skillset: st-multimodal-pdf-skillset)
✅ st-multimodal-pptx-indexer — 등록됨 (skillset: st-multimodal-pptx-skillset)
✅ st-multimodal-verbalized-indexer — 등록됨 (skillset: st-multimodal-verbalized-skillset)


## 4. 인덱서 실행

두 인덱서를 순차적으로 실행합니다.  
- **[B] Basic**: DI Layout → SplitSkill (markdown) → Embedding (빠름, ~2-5분/PDF)
- **[C] Verbalized**: DI Layout → GPT-5.4 Verbalize → Split → Embedding (느림, ~5-15분/PDF)

> Basic 인덱서를 먼저 실행하고 완료 후 Verbalized를 실행합니다 (리소스 경합 방지).

In [14]:
def run_indexer(indexer_name: str) -> bool:
    """인덱서 실행"""
    url = f'{SEARCH_ENDPOINT}/indexers/{indexer_name}/run?api-version={API_VERSION}'
    resp = requests.post(url, headers=get_search_headers())
    if resp.status_code == 202:
        print(f'  ▶ {indexer_name} 실행 시작')
        return True
    else:
        print(f'  ✗ {indexer_name} 실행 실패: {resp.status_code} {resp.text[:200]}')
        return False


def get_indexer_status(indexer_name: str) -> dict:
    """인덱서 상태 조회"""
    url = f'{SEARCH_ENDPOINT}/indexers/{indexer_name}/status?api-version={API_VERSION}'
    resp = requests.get(url, headers=get_search_headers())
    return resp.json()


def _last_start_time(status: dict) -> str | None:
    last = status.get('lastResult') or {}
    return last.get('startTime')


def wait_indexer(indexer_name: str, timeout_sec: int = 1200, poll_interval: int = 15):
    """인덱서 완료 대기.

    주의: 이전 run 의 lastResult 가 남아있을 수 있으므로,
    호출 시점의 lastResult.startTime 을 기준선으로 잡고
    그보다 새 startTime 의 lastResult 가 success/failure 로 끝날 때까지 대기한다.
    또한 top-level status 가 'running' 이면 계속 대기한다.
    """
    baseline = _last_start_time(get_indexer_status(indexer_name))
    start = time.time()
    print(f'  ⏳ {indexer_name} 완료 대기 중... (baseline lastStart={baseline})')
    while True:
        status = get_indexer_status(indexer_name)
        top_state = status.get('status', 'unknown')  # 'running'|'error'|'success'|...
        last = status.get('lastResult') or {}
        last_state = last.get('status', 'unknown')
        last_start = last.get('startTime')
        processed = last.get('itemsProcessed', 0)
        failed = last.get('itemsFailed', 0)

        elapsed = int(time.time() - start)
        is_new = (last_start is not None and last_start != baseline)
        print(f'    [{elapsed:>4d}s] top={top_state} last={last_state} new={is_new} '
              f'processed={processed} failed={failed}')

        # 새 run 의 결과가 terminal 상태로 도달했고, top-level 도 running 아님
        if is_new and last_state in ('success', 'transientFailure', 'persistentFailure') \
                and top_state != 'running':
            return last_state, last
        if elapsed > timeout_sec:
            print(f'    ⚠ timeout ({timeout_sec}s)')
            return 'timeout', last
        time.sleep(poll_interval)


In [15]:
# # ── [B-PDF] Basic PDF 인덱서 실행 ──────────────────────────────
# print('='*60)
# print('[B-PDF] Basic PDF 인덱서 실행')
# print('    DI Layout → markdown_split → Embedding')
# print('='*60)

# if run_indexer(PDF_INDEXER):
#     time.sleep(5)
#     state, result = wait_indexer(PDF_INDEXER)
#     print(f'\n  결과: {state}')
#     if result:
#         print(f'  처리: {result.get("itemsProcessed", 0)}개, 실패: {result.get("itemsFailed", 0)}개')
#         if result.get('errors'):
#             for err in result['errors'][:3]:
#                 print(f'    ERROR: {err.get("message", "")[:200]}')

# # ── [B-PPTX] Basic PPTX 인덱서 실행 ────────────────────────────
# print()
# print('='*60)
# print('[B-PPTX] Basic PPTX 인덱서 실행')
# print('    DI Layout → pptx_page_split → Embedding')
# print('='*60)

# if run_indexer(PPTX_INDEXER):
#     time.sleep(5)
#     state, result = wait_indexer(PPTX_INDEXER)
#     print(f'\n  결과: {state}')
#     if result:
#         print(f'  처리: {result.get("itemsProcessed", 0)}개, 실패: {result.get("itemsFailed", 0)}개')
#         if result.get('errors'):
#             for err in result['errors'][:3]:
#                 print(f'    ERROR: {err.get("message", "")[:200]}')


In [16]:
# # ── [C] Verbalized 인덱서 실행 ─────────────────────────────────
# print('='*60)
# print('[C] Verbalized 인덱서 실행')
# print('    DI Layout → GPT-5.4 Verbalize → Markdown Split → Embedding')
# print('='*60)

# if run_indexer(VERBALIZED_INDEXER):
#     time.sleep(5)
#     state, result = wait_indexer(VERBALIZED_INDEXER, timeout_sec=1800, poll_interval=20)
#     print(f'\n  결과: {state}')
#     if result:
#         print(f'  처리: {result.get("itemsProcessed", 0)}개, 실패: {result.get("itemsFailed", 0)}개')
#         if result.get('errors'):
#             for err in result['errors'][:3]:
#                 print(f'    ERROR: {err.get("message", "")[:200]}')

## 5. 인덱싱 결과 확인

두 인덱스에 저장된 문서 수와 통계를 비교합니다.

In [17]:
def get_index_stats(index_name: str) -> dict:
    url = f'{SEARCH_ENDPOINT}/indexes/{index_name}/stats?api-version={API_VERSION}'
    resp = requests.get(url, headers=get_search_headers())
    return resp.json() if resp.status_code == 200 else {}

# print(f'{"인덱스":<40} {"문서 수":>10} {"저장 크기":>15}')
# print('-'*70)

# for idx_name in [PDF_INDEX, PPTX_INDEX, VERBALIZED_INDEX]:
#     stats = get_index_stats(idx_name)
#     doc_count = stats.get('documentCount', 'N/A')
#     storage = stats.get('storageSize', 0)
#     storage_str = f'{storage:,} bytes' if isinstance(storage, int) else 'N/A'
#     print(f'{idx_name:<40} {doc_count:>10} {storage_str:>15}')


In [18]:
# # ── 인덱서 상세 상태 (최근 실행) ─────────────────────────────
# for indexer_name in [PDF_INDEXER, PPTX_INDEXER, VERBALIZED_INDEXER]:
#     status = get_indexer_status(indexer_name)
#     last = status.get('lastResult') or {}
#     print(f'\n{indexer_name}:')
#     print(f'  상태: {last.get("status", "N/A")}')
#     print(f'  처리: {last.get("itemsProcessed", 0)}개')
#     print(f'  실패: {last.get("itemsFailed", 0)}개')
#     print(f'  시작: {last.get("startTime", "N/A")}')
#     print(f'  종료: {last.get("endTime", "N/A")}')
#     if last.get('errors'):
#         print(f'  오류:')
#         for err in last['errors'][:5]:
#             print(f'    - {err.get("message", "")[:150]}')


## 6. 샘플 문서 확인 (Quick Look)

각 인덱스에서 첫 3개 문서를 조회하여 인덱싱 품질을 빠르게 확인합니다.

In [19]:
# def search_sample(index_name: str, top: int = 3) -> list:
#     """인덱스에서 샘플 문서 조회"""
#     url = f'{SEARCH_ENDPOINT}/indexes/{index_name}/docs/search?api-version={API_VERSION}'
#     payload = {
#         'search': '*',
#         'top': top,
#         'select': 'id,source_file_name,content_type,content',
#     }
#     resp = requests.post(url, headers=get_search_headers(), json=payload)
#     if resp.status_code == 200:
#         return resp.json().get('value', [])
#     return []

# for label, idx in [('B-PDF', PDF_INDEX), ('B-PPTX', PPTX_INDEX), ('C-Verbalized', VERBALIZED_INDEX)]:
#     print('='*60)
#     print(f'[{label}] {idx}')
#     print('='*60)
#     for doc in search_sample(idx):
#         print(f'\n  📄 {doc.get("source_file_name", "?")} [{doc.get("content_type", "?")}]')
#         content = doc.get('content', '') or ''
#         print(f'     {content[:200]}...' if len(content) > 200 else f'     {content}')
#     print()


## 7. Indexer Caching 효과 측정 — 멀티모달 파이프라인

[scripts/setup_ai_search_multimodal_pipeline.py](../scripts/setup_ai_search_multimodal_pipeline.py) 의 `SETUP_ENABLE_CACHE=1` 옵션을 활성화하면 멀티모달 indexer 정의에 `cache` 블록이 추가되어 **incremental enrichment cache** 가 켜집니다.

### 왜 멀티모달에서 cache 효과가 큰가?

노트북 03 (텍스트 임베딩 전용) 의 §6 결과와 비교하면 차이가 명확합니다.

| 스킬 | 1건 처리 시간 | API 비용 (1K 호출) | cache HIT 의 storage lookup 비용 | 효과 |
|---|---|---|---|---|
| `AzureOpenAIEmbeddingSkill` (text-embedding-3-large, batch=16) | ~5 ms / 문서 | $0.13/1M tokens | ~30–150 ms / 문서 | **❌ 시간상 손해** (notebook 03) |
| `DocumentIntelligenceLayoutSkill` (Layout) | **2–10 s / 페이지** | ~$0.015 / 페이지 | ~30–150 ms / 페이지 | **✅ 시간·비용 모두 이득** |
| `ChatCompletionSkill` (GPT-4o verbalize) | **3–8 s / 이미지** | ~$0.005–0.020 / 호출 | ~30–150 ms / 호출 | **✅ 시간·비용 모두 이득** |

→ 멀티모달 indexer 는 DI Layout 과 GPT verbalize 가 dominant skill 이므로 **cache HIT 한 건당 수 초 단축**이 가능하고, 그만큼 storage lookup 오버헤드 (~50 ms) 가 묻혀버립니다.

### 비교 시나리오 (대상: `st-multimodal-verbalized-indexer` — 가장 비싼 verbalized 파이프라인)

| | 시나리오 | 동작 | cache 상태 | 예상 |
|---|---|---|---|---|
| **A** | 캐시 OFF reindex | setup script 호출 (`SETUP_ENABLE_CACHE=0`) → 자동 reset + run | — | baseline (DI + GPT 전부 재호출) |
| **B** | 캐시 ON reindex (1차) | setup script 호출 (`SETUP_ENABLE_CACHE=1`) → 자동 reset + run | 비어있음→채워짐 | A 와 유사 (캐시 채우는 단계) |
| **C** | 캐시 ON reindex (2차) — **재사용 검증** | setup script 호출 (`SETUP_ENABLE_CACHE=1`) → 자동 reset + run | warm | A 대비 큰 폭 단축 (DI/GPT cache HIT) |

> Verbalized 인덱서는 1개 파일당 5–15분 걸리므로 실험 데이터는 **소량 (1–3개 PDF)** 만 두고 진행하세요. 큰 데이터로는 1회 reindex 만으로도 30분 이상 걸릴 수 있습니다.


In [20]:
# 실험 헬퍼: setup_ai_search_multimodal_pipeline.py 를 cache 토글하여 호출
# (스크립트가 indexer 정의를 (재)생성 → reset → run → 완료대기 순으로 진행)
from datetime import datetime

EXP_INDEXER_MM = VERBALIZED_INDEXER   # 대상: verbalized (DI Layout + GPT verbalize)
EXP_INDEX_MM   = VERBALIZED_INDEX
EXP_SOURCE_MM  = SOURCE                # 'st'

def reset_indexer(indexer_name: str) -> bool:
    """indexer reset (다음 run 에서 전 문서 재처리). cache 가 켜져 있으면 변경 없는 skill 출력은 cache HIT."""
    url = f"{SEARCH_ENDPOINT}/indexers/{indexer_name}/reset?api-version={API_VERSION}"
    r = requests.post(url, headers=get_search_headers())
    return r.status_code in (204, 200)


def wait_until_idle(indexer_name: str, timeout_sec: int = 7200, poll_interval: int = 30) -> str:
    """현재 진행 중인(혹은 스케줄된) run 이 모두 끝나 idle 상태가 될 때까지 대기."""
    t0 = time.time()
    while True:
        st = get_indexer_status(indexer_name)
        top = st.get('status', 'unknown')
        last = (st.get('executionHistory') or [{}])[0]
        if top != 'running' and last.get('status') != 'inProgress':
            return top
        if time.time() - t0 > timeout_sec:
            print(f"  ⚠ wait_until_idle timeout (top={top})")
            return 'timeout'
        time.sleep(poll_interval)


def run_mm_setup(cache_on: bool, do_reset: bool = True, timeout: int = 7200) -> tuple[int, float, str]:
    """멀티모달 setup 스크립트 호출. cache_on=True → SETUP_ENABLE_CACHE=1 주입.

    do_reset=True 면 setup 후 indexer reset → 재실행하여 cache 효과를 측정.
    timeout: wait_indexer 의 polling 한계. verbalized 인덱서는 ~50분 걸리므로 넉넉히 2시간.
    """
    env = os.environ.copy()
    env["SETUP_ENABLE_CACHE"] = "1" if cache_on else "0"
    t0 = time.time()
    # 0) 다른 invocation 이 진행 중이면 끝날 때까지 대기 (409 회피)
    idle = wait_until_idle(EXP_INDEXER_MM, timeout_sec=timeout)
    print(f"  pre-idle state = {idle}")
    # 1) indexer 정의 (재)등록 (cache 옵션 반영). --run-indexer 없이 생성만.
    subprocess.run(
        [sys.executable, "scripts/setup_ai_search_multimodal_pipeline.py",
         "--source", EXP_SOURCE_MM],
        cwd=os.path.abspath(".."),
        capture_output=True, text=True, encoding="utf-8", errors="replace",
        timeout=timeout, env=env, check=False,
    )
    # 2) reset → 전 문서 재처리 (cache HIT 검증용)
    if do_reset:
        reset_indexer(EXP_INDEXER_MM)
        time.sleep(2)
    # 3) run + wait
    if not run_indexer(EXP_INDEXER_MM):
        return 1, time.time() - t0, "run_indexer failed"
    time.sleep(3)
    state, last = wait_indexer(EXP_INDEXER_MM, timeout_sec=timeout, poll_interval=20)
    elapsed = time.time() - t0
    summary = f"state={state} processed={last.get('itemsProcessed', 0)} failed={last.get('itemsFailed', 0)}"
    return (0 if state == "success" else 2), elapsed, summary


def get_indexer_metrics(indexer_name: str) -> dict:
    """마지막 indexer 실행의 itemsProcessed/Failed + 소요시간."""
    st = get_indexer_status(indexer_name)
    last = st.get("lastResult") or {}
    start_t = last.get("startTime")
    end_t   = last.get("endTime")
    elapsed_indexer = None
    if start_t and end_t:
        s = datetime.fromisoformat(start_t.replace("Z", "+00:00"))
        e = datetime.fromisoformat(end_t.replace("Z", "+00:00"))
        elapsed_indexer = (e - s).total_seconds()
    return {
        "status": last.get("status"),
        "items_processed": last.get("itemsProcessed", 0),
        "items_failed": last.get("itemsFailed", 0),
        "indexer_elapsed_sec": elapsed_indexer,
        "start_time": start_t,
        "end_time": end_t,
    }

print(f"실험 대상 indexer = {EXP_INDEXER_MM}")
print(f"실험 대상 index   = {EXP_INDEX_MM}")
print(f"Search Endpoint   = {SEARCH_ENDPOINT}")


실험 대상 indexer = st-multimodal-verbalized-indexer
실험 대상 index   = st-multimodal-verbalized-index
Search Endpoint   = https://search-ragi-63325wdo.search.windows.net


In [ ]:
# 시나리오 A 는 skip. 대신 B/C 시작 전에 **모든 캐시·인덱서·인덱스 초기화**.
# 1) 진행 중 invocation 종료 대기
# 2) indexer 삭제 (cache 메타 + cache blob 참조 제거)
# 3) search index 삭제 (모든 문서 제거)
# 4) setup script 재실행 (cache OFF) → indexer/index 새로 생성  (cache id 가 새로 발급)
# → 이후 B 가 cache 를 채우는 첫 run, C 가 cache HIT 검증 run 이 됨.

print("="*70, "\n[RESET] 모든 cache / indexer / index 초기화", "\n"+"="*70)
print(f"시작: {datetime.now().strftime('%H:%M:%S')}")

# 1) 진행 중 run 종료 대기
idle = wait_until_idle(EXP_INDEXER_MM, timeout_sec=7200)
print(f"  pre-idle = {idle}")

# 2) indexer 삭제
for _name in (EXP_INDEXER_MM,):
    r = requests.delete(f"{SEARCH_ENDPOINT}/indexers/{_name}?api-version={API_VERSION}",
                        headers=get_search_headers())
    print(f"  DELETE indexer {_name} → {r.status_code}")

# 3) search index 삭제 (문서 전부 제거)
for _name in (EXP_INDEX_MM,):
    r = requests.delete(f"{SEARCH_ENDPOINT}/indexes/{_name}?api-version={API_VERSION}",
                        headers=get_search_headers())
    print(f"  DELETE index   {_name} → {r.status_code}")

# 4) setup script 재실행 (cache OFF 로 재생성 → 이후 B 가 cache ON 으로 덮어씀)
env = os.environ.copy()
env["SETUP_ENABLE_CACHE"] = "0"
res = subprocess.run(
    [sys.executable, "scripts/setup_ai_search_multimodal_pipeline.py",
     "--source", EXP_SOURCE_MM],
    cwd=os.path.abspath(".."),
    capture_output=True, text=True, encoding="utf-8", errors="replace",
    timeout=600, env=env, check=False,
)
print("\n".join(res.stdout.splitlines()[-20:]))
print(f"  setup rc={res.returncode}")

# 메트릭 변수 초기화 (A skip 표시)
metrics_a = {"indexer_elapsed_sec": None}
rc_a_mm, wall_a_mm, log_a_mm = -1, 0.0, "(scenario A skipped)"
print("\n[RESET 완료] 이제 B(cache 채움) → C(cache HIT) 만 측정합니다.")



[RESET] 모든 cache / indexer / index 초기화 
시작: 08:24:12


In [20]:
# 시나리오 B: cache ON reindex (1차 — 캐시가 채워짐) — 시간상 A 와 비슷, cache 가 storage 에 적재됨
print("="*70, f"\nB. SETUP_ENABLE_CACHE=1  →  멀티모달 reindex (캐시 채움)", "\n"+"="*70)
print(f"시작: {datetime.now().strftime('%H:%M:%S')}")

rc_b_mm, wall_b_mm, log_b_mm = run_mm_setup(cache_on=True)

tail = "\n".join(log_b_mm.splitlines()[-30:])
print(tail)

metrics_b = get_indexer_metrics(EXP_INDEXER_MM)
print(f"\n[B 결과] rc={rc_b_mm}  wall={wall_b_mm:.1f}s  indexer={metrics_b['indexer_elapsed_sec']}s "
      f"items={metrics_b['items_processed']}/failed={metrics_b['items_failed']}")


B. SETUP_ENABLE_CACHE=1  →  멀티모달 reindex (캐시 채움) 
시작: 16:22:55


  ▶ st-multimodal-verbalized-indexer 실행 시작
  ⏳ st-multimodal-verbalized-indexer 완료 대기 중... (baseline lastStart=2026-05-18T16:23:40.329Z)
    [   1s] top=running last=inProgress new=False processed=0 failed=0
    [  22s] top=running last=inProgress new=False processed=0 failed=0
    [  43s] top=running last=inProgress new=False processed=0 failed=0
    [  64s] top=running last=inProgress new=False processed=0 failed=0
    [  85s] top=running last=inProgress new=False processed=0 failed=0
    [ 107s] top=running last=inProgress new=False processed=0 failed=0
    [ 128s] top=running last=inProgress new=False processed=4 failed=0
    [ 149s] top=running last=inProgress new=False processed=4 failed=0
    [ 170s] top=running last=inProgress new=False processed=4 failed=0
    [ 191s] top=running last=inProgress new=False processed=4 failed=0
    [ 213s] top=running last=inProgress new=False processed=4 failed=0
    [ 234s] top=running last=inProgress new=False processed=4 failed=0
    [ 255s]

In [21]:
# 시나리오 C: cache ON reindex (2차 — 캐시 재사용) — 핵심 측정. DI/GPT 호출은 cache HIT 으로 skip 되어 큰 폭 단축 기대
print("="*70, f"\nC. SETUP_ENABLE_CACHE=1  →  멀티모달 reindex (2차, 캐시 HIT)", "\n"+"="*70)
print(f"시작: {datetime.now().strftime('%H:%M:%S')}")

rc_c_mm, wall_c_mm, log_c_mm = run_mm_setup(cache_on=True)

tail = "\n".join(log_c_mm.splitlines()[-30:])
print(tail)

metrics_c = get_indexer_metrics(EXP_INDEXER_MM)
print(f"\n[C 결과] rc={rc_c_mm}  wall={wall_c_mm:.1f}s  indexer={metrics_c['indexer_elapsed_sec']}s "
      f"items={metrics_c['items_processed']}/failed={metrics_c['items_failed']}")

# B 대비 단축률 (A 는 skip)
if metrics_b.get("indexer_elapsed_sec") and metrics_c.get("indexer_elapsed_sec"):
    diff = metrics_b["indexer_elapsed_sec"] - metrics_c["indexer_elapsed_sec"]
    pct  = diff / metrics_b["indexer_elapsed_sec"] * 100
    print(f"\n  → B (cache 채움) 대비 C (cache HIT): {diff:+.1f}s ({pct:+.1f}%) "
          f"{'단축 (DI/GPT cache HIT 효과)' if diff > 0 else '단축 없음/오히려 증가'}")


C. SETUP_ENABLE_CACHE=1  →  멀티모달 reindex (2차, 캐시 HIT) 
시작: 23:48:07
  ▶ st-multimodal-verbalized-indexer 실행 시작
  ⏳ st-multimodal-verbalized-indexer 완료 대기 중... (baseline lastStart=2026-05-18T23:49:12.341Z)
    [   1s] top=running last=inProgress new=False processed=0 failed=0
    [  22s] top=running last=inProgress new=False processed=0 failed=0
    [  43s] top=running last=inProgress new=False processed=0 failed=0
    [  64s] top=running last=inProgress new=False processed=0 failed=0
    [  85s] top=running last=inProgress new=False processed=0 failed=0
    [ 106s] top=running last=inProgress new=False processed=0 failed=0
    [ 128s] top=running last=inProgress new=False processed=1 failed=0
    [ 149s] top=running last=inProgress new=False processed=1 failed=0
    [ 170s] top=running last=inProgress new=False processed=1 failed=0
    [ 191s] top=running last=inProgress new=False processed=6 failed=0
    [ 212s] top=running last=inProgress new=False processed=6 failed=0
    [ 233s] to

In [ ]:
# 요약: A/B/C 비교 표 + REPORT_CACHE_MM.md 저장
import pandas as pd

rows = [
    {"scenario": "A. cache OFF (baseline)",         "wall_sec": round(wall_a_mm, 1),
     "indexer_sec": metrics_a.get("indexer_elapsed_sec"),
     "items": metrics_a.get("items_processed"), "failed": metrics_a.get("items_failed")},
    {"scenario": "B. cache ON (1st — fill cache)",  "wall_sec": round(wall_b_mm, 1),
     "indexer_sec": metrics_b.get("indexer_elapsed_sec"),
     "items": metrics_b.get("items_processed"), "failed": metrics_b.get("items_failed")},
    {"scenario": "C. cache ON (2nd — HIT)",          "wall_sec": round(wall_c_mm, 1),
     "indexer_sec": metrics_c.get("indexer_elapsed_sec"),
     "items": metrics_c.get("items_processed"), "failed": metrics_c.get("items_failed")},
]
df_cache_mm = pd.DataFrame(rows)
display(df_cache_mm)

# 단축 효과 계산
base = metrics_a.get("indexer_elapsed_sec") or 0
hit  = metrics_c.get("indexer_elapsed_sec") or 0
saving_sec = base - hit
saving_pct = (saving_sec / base * 100) if base else 0

md = [
    "# Multimodal Indexer Caching — 실험 결과",
    "",
    f"- 대상 indexer: `{EXP_INDEXER_MM}`",
    f"- 대상 index  : `{EXP_INDEX_MM}`",
    f"- source      : `{EXP_SOURCE_MM}`",
    "",
    "## 시나리오별 소요시간",
    "",
    df_cache_mm.to_markdown(index=False),
    "",
    "## Cache HIT 효과 (C vs A)",
    "",
    f"- baseline (A) indexer elapsed : **{base:.1f} s**" if base else "- baseline 측정 실패",
    f"- cache HIT (C) indexer elapsed: **{hit:.1f} s**" if hit else "- cache HIT 측정 실패",
    f"- 단축: **{saving_sec:+.1f} s ({saving_pct:+.1f}%)**" if base else "",
    "",
    "## 해석",
    "",
    "- 멀티모달 파이프라인은 `DocumentIntelligenceLayoutSkill` 과 GPT verbalize `ChatCompletionSkill` 이 dominant cost (각각 수 초/호출).",
    "- Cache HIT 시 두 skill 호출이 Storage Table/Blob 조회 (~50 ms) 로 대체되므로 **시간·비용 모두 큰 폭 절감**.",
    "- 노트북 03 (텍스트 임베딩) 의 결과와 비교: 텍스트는 batch 임베딩이 ~5 ms/문서로 cache lookup 보다 빨라 오히려 손해. 멀티모달에서는 반대로 cache 가 명확한 이득.",
]
report_path = Path("REPORT_CACHE_MM.md")
report_path.write_text("\n".join(md), encoding="utf-8")
print(f"\n저장: {report_path.resolve()}")


---

## 정리

| 단계 | 설명 |
|------|------|
| PDF/PPTX 업로드 | Blob Storage `raw-documents/raw/pdf/st/` 에 업로드 |
| [B] Basic 인덱서 실행 | DI Layout → SplitSkill(markdown) → Embedding |
| [C] Verbalized 인덱서 실행 | DI Layout → GPT-5.4 Verbalize → Split → Embedding |
| 결과 확인 | 문서 수, 청크 품질 비교 |

### 다음 단계
→ [06-multimodal-search.ipynb](06-multimodal-search.ipynb) — 두 인덱스의 검색 품질 비교 및 RAG 실험